# 🎬 VideoRAG — Orchestrator · Worker · Evaluator

```
User Question
     │
     ▼
┌──────────────────────────────────┐
│          ORCHESTRATOR            │  ← Gemini LLM
│  Routes: audio-rich OR           │
│          image-rich path         │
└────────┬─────────────────────────┘
         │  retrieval tool calls
    ┌────┴────────────────────────────────┐
    │              WORKERS                │
    │  • retrieve_audio_transcript        │  FAISS + Cohere embed
    │  • retrieve_image_captions          │  FAISS + Cohere embed
    └────────────┬────────────────────────┘
                 │
    ┌────────────▼────────────────────────┐
    │            EVALUATOR                │  ← Gemini Flash
    │  PASS  → return answer              │
    │  RETRY → inject hint, loop again    │
    │  UNANSWERABLE → graceful exit       │
    └─────────────────────────────────────┘
```
**Routing logic**: if ≥ 30 % of audio chunks have non-empty transcripts → audio-rich path; otherwise → image-rich path.
**Tool policy**: use only `retrieve_audio_transcript` and `retrieve_image_captions`; each may be called at most twice per answer.


---
## Part 1 — Extraction (original code)
Whisper ASR · BLIP captioning · EasyOCR

In [1]:
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
)


/home/es22btech11034/miniconda3/envs/vdo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from transformers import pipeline

import os

from moviepy import VideoFileClip
import easyocr
import faiss
import cohere
import numpy as np
from PIL import Image
from IPython.display import HTML
from base64 import b64encode
import IPython
from tqdm import tqdm
import gc

In [3]:
VIDEO_PATH = "./videos/periodic_table.mp4"  # Video file path
CLIPS_DIR = "audio_chunk"              # Audio clips storage folder
AUDIO_CHUNK_DURATION = 30              # Interval for audio clip chunking
VIDEO_CHUNK_DURATION = 30              # Interval for frame caption generation
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [4]:
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    chunk_length_s=AUDIO_CHUNK_DURATION,
    dtype=torch.float16,
    device=DEVICE,
)


BATCH_SIZE = 4

os.makedirs(CLIPS_DIR, exist_ok=True)

# ── Load video ───────────────────────────────────────────────────

video = VideoFileClip(VIDEO_PATH)
video_duration = int(video.duration)

print(f"Video Duration: {video_duration} seconds")

clip_paths = []

# ── Extract audio clips ──────────────────────────────────────────

for start_time in tqdm(
    range(0, video_duration, AUDIO_CHUNK_DURATION),
    desc="Extracting audio clips"
):
    end_time = min(start_time + AUDIO_CHUNK_DURATION, video_duration)

    clip = video.subclipped(start_time, end_time)

    audio_clip_filename = os.path.join(
        CLIPS_DIR,
        f"clip_{start_time}_{end_time}.mp3"
    )

    clip.audio.write_audiofile(
        audio_clip_filename,
        logger=None
    )

    clip_paths.append((start_time, audio_clip_filename))

video.close()

# ── Batched Whisper transcription ────────────────────────────────

transcript = []

for i in tqdm(
    range(0, len(clip_paths), BATCH_SIZE),
    desc="Transcribing batches"
):
    batch = clip_paths[i:i + BATCH_SIZE]

    batch_start_times = [x[0] for x in batch]
    batch_audio_paths = [x[1] for x in batch]

    results = pipe(
        batch_audio_paths,
        batch_size=BATCH_SIZE,
        return_timestamps=False
    )

    for start_time, result in zip(batch_start_times, results):
        transcript.append({
            "start_time": start_time,
            "text": result["text"]
        })

# ── Cleanup ──────────────────────────────────────────────────────

for _, clip_path in tqdm(
    clip_paths,
    desc="Cleaning up"
):
    os.remove(clip_path)

os.rmdir(CLIPS_DIR)

del pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

Loading weights: 100%|██████████| 947/947 [00:00<00:00, 1694.95it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Video Duration: 867 seconds


Transcribing batches:   0%|          | 0/8 [00:00<?, ?it/s][transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generati

In [5]:
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(DEVICE)


# Initialize EasyOCR
reader = easyocr.Reader(['en'])

clip = VideoFileClip(VIDEO_PATH)
duration = int(clip.duration)

results = []

for t in range(0, duration, VIDEO_CHUNK_DURATION):

    # -------------------------
    # GET FRAME
    # -------------------------
    frame = clip.get_frame(t)  # RGB numpy array

    pil_image = Image.fromarray(frame).convert("RGB")

    # -------------------------
    # IMAGE CAPTIONING
    # -------------------------
    inputs = processor(
        pil_image,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(**inputs)

    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    # -------------------------
    # OCR WITH EASYOCR
    # -------------------------
    ocr_result = reader.readtext(frame)

    extracted_texts = []

    for item in ocr_result:
        text = item[1]
        confidence = item[2]

        # Optional confidence filter
        if confidence > 0.3:
            extracted_texts.append(text)

    # -------------------------
    # STORE
    # -------------------------
    results.append({
        "timestamp": t,
        "caption": caption,
        "ocr": extracted_texts
    })

clip.close()


del model
del processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

Loading weights: 100%|██████████| 616/616 [00:00<00:00, 4138.47it/s]
/home/es22btech11034/miniconda3/envs/vdo/lib/python3.11/site-packages/transformers/generation/utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


---
## Part 2 — Agent Setup

### 2.1 API keys & agent config

In [6]:
import json
import io
import base64
import re
from collections import Counter
from typing import Any

# import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

GOOGLE_API_KEY  = os.environ.get("GOOGLE_API_KEY",  "YOUR_GOOGLE_API_KEY")
COHERE_API_KEY  = os.environ.get("COHERE_API_KEY",  "YOUR_COHERE_API_KEY")
GROQ_API_KEY    = os.environ.get("GROQ_API_KEY",    "YOUR_GROQ_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
# genai.configure(api_key=GOOGLE_API_KEY)

co = cohere.Client(COHERE_API_KEY)

# ── Routing threshold ─────────────────────────────────────────────────────────
AUDIO_RICH_THRESHOLD = 0.30   # fraction of chunks that must contain valid English
FRAME_WINDOW_SEC     = 10     # ± seconds around a timestamp for live sampling
MAX_EVAL_RETRIES     = 0  # no retries — evaluator is observability only

print("Config OK")

Config OK


### 2.2 Cohere FAISS indexes

In [7]:
def cohere_embed(texts: list[str], input_type: str = "search_document") -> np.ndarray:
    """Embed a list of texts with Cohere embed-english-v3.0."""
    response = co.embed(
        texts=texts,
        model="embed-english-v3.0",
        input_type=input_type,
    )
    return np.array(response.embeddings, dtype="float32")


def build_faiss_index(texts: list[str], metadatas: list[dict]):
    """Return (faiss_index, texts, metadatas) or (None, [], []) if empty."""
    if not texts:
        return None, [], []
    vecs = cohere_embed(texts)
    faiss.normalize_L2(vecs)
    idx = faiss.IndexFlatIP(vecs.shape[1])   # inner-product == cosine after L2-norm
    idx.add(vecs)
    return idx, texts, metadatas


def faiss_search(query: str, index, texts, metadatas, k: int = 3) -> list[dict]:
    if index is None:
        return []
    # Enforce a maximum of 10 documents per retrieval call.
    k = max(0, min(int(k), 10))
    if k == 0:
        return []
    q = cohere_embed([query], input_type="search_query")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)
    return [
        {"score": float(scores[0][i]), "text": texts[ids[0][i]], "meta": metadatas[ids[0][i]]}
        for i in range(k) if ids[0][i] >= 0
    ]


COMMON_ENGLISH_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "because", "but", "by",
    "for", "from", "had", "has", "have", "he", "her", "his", "i", "if",
    "in", "is", "it", "its", "like", "my", "not", "of", "on", "or",
    "our", "she", "that", "the", "their", "there", "they", "this", "to",
    "was", "we", "were", "what", "when", "where", "which", "who", "will",
    "with", "you", "your",
}


def is_valid_english_transcript(text: str) -> bool:
    """Return True when the transcript chunk looks like real English."""
    if not text or not text.strip():
        return False

    words = re.findall(r"[a-z]+", text.lower())
    if len(words) < 4:
        return False

    alpha_chars = sum(ch.isalpha() for ch in text)
    alpha_ratio = alpha_chars / max(len(text.replace(" ", "")), 1)
    if alpha_ratio < 0.7:
        return False

    counts = Counter(words)
    if counts.most_common(1)[0][1] / len(words) > 0.75:
        return False

    if not any(word in COMMON_ENGLISH_WORDS for word in words):
        return False

    return True


def english_transcript_ratio(records: list[dict]) -> float:
    if not records:
        return 0.0
    return sum(1 for row in records if is_valid_english_transcript(row.get("text", ""))) / len(records)


# ── Build audio index ─────────────────────────────────────────────────────────
valid_transcript_rows = [row for row in transcript if is_valid_english_transcript(row.get("text", ""))]
audio_texts = [
    f"[t={row['start_time']}s] {row['text']}"
    for row in valid_transcript_rows
]
audio_metas = [
    {"start_time": row["start_time"]}
    for row in valid_transcript_rows
]
audio_index, audio_texts_store, audio_meta_store = build_faiss_index(audio_texts, audio_metas)

# ── Build image index ─────────────────────────────────────────────────────────
img_texts = [
    f"[t={r['timestamp']}s] caption: {r['caption']}. ocr: {' '.join(r['ocr'])}"
    for r in results
]
img_metas = [{"timestamp": r["timestamp"]} for r in results]
img_index, img_texts_store, img_meta_store = build_faiss_index(img_texts, img_metas)

print(f"Audio chunks indexed : {len(audio_texts)}")
print(f"Image chunks indexed : {len(img_texts)}")
print(f"Valid English audio  : {english_transcript_ratio(transcript):.0%}")

# ── Routing decision ──────────────────────────────────────────────────────────
AUDIO_RICH = english_transcript_ratio(transcript) >= AUDIO_RICH_THRESHOLD
print(f"\nAudio richness : {english_transcript_ratio(transcript):.0%}")
print(f"Active path    : {'AUDIO-RICH' if AUDIO_RICH else 'IMAGE-RICH'}")

Audio chunks indexed : 29
Image chunks indexed : 29
Valid English audio  : 100%

Audio richness : 100%
Active path    : AUDIO-RICH


### 2.3 Worker tools

In [8]:

# ── TOOL 1: retrieve_audio_transcript ────────────────────────────────────────
@tool
def retrieve_audio_transcript(query: str, k: int = 3) -> str:
    """
    Search the Whisper-transcribed audio chunks with Cohere embeddings.
    Returns the top-k most relevant chunks as JSON:
      [{"start_time_sec": int, "text": str, "score": float}]
    Use this tool when audio transcripts are available (audio-rich video).
    """
    hits = faiss_search(query, audio_index, audio_texts_store, audio_meta_store, k=k)
    out = [
        {"start_time_sec": h["meta"]["start_time"], "text": h["text"], "score": round(h["score"], 3)}
        for h in hits
    ]
    return json.dumps(out, indent=2)


    # ── TOOL 2: retrieve_image_captions ─────────────────────────────────────────
@tool
def retrieve_image_transcript(query: str, k: int = 3) -> str:
    """
    Search the pre-computed BLIP captions + EasyOCR text with Cohere embeddings.
    Returns the top-k most relevant frames as JSON:
      [{"timestamp_sec": int, "caption_ocr": str, "score": float}]
    Use this tool when audio is sparse or absent.
    """
    hits = faiss_search(query, img_index, img_texts_store, img_meta_store, k=k)
    out = [
        {"timestamp_sec": h["meta"]["timestamp"], "caption_ocr": h["text"], "score": round(h["score"], 3)}
        for h in hits
    ]
    return json.dumps(out, indent=2)



# ── TOOL 3: caption_frame ────────────────────────────────────────────────────
@tool
def caption_frame(t: int) -> str:
    """
    Extract frames at timestamp t and t+5 seconds from the video, then generate
    a detailed visual caption by passing both frames as images to a multimodal LLM.

    Use this tool ONLY when:
      - The question explicitly references a visual moment or timestamp
        (e.g. "what is shown at 23s?", "what does the screen look like at 1:30?"), OR
      - The audio transcript and image-caption indexes do not contain sufficient
        information to answer the question and a direct visual inspection is needed.

    Args:
        t: Timestamp in seconds. Frames at t and t+5 will be captured.

    Returns:
        A JSON string: {"timestamp_sec": t, "caption": <multimodal LLM description>}
    """
    import io
    import base64

    clip = VideoFileClip(VIDEO_PATH)
    video_duration = clip.duration

    timestamps = [t, min(t + 5, video_duration)]
    b64_images = []

    for ts in timestamps:
        frame = clip.get_frame(ts)          # RGB numpy array
        pil_img = Image.fromarray(frame).convert("RGB")
        buf = io.BytesIO()
        pil_img.save(buf, format="JPEG")
        b64_images.append(base64.b64encode(buf.getvalue()).decode("utf-8"))

    clip.close()

    # Build a multimodal message: two frames + instruction
    content = [
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64_images[0]}"},
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64_images[1]}"},
        },
        {
            "type": "text",
            "text": (
                f"These are two consecutive frames from a video at t={timestamps[0]}s "
                f"and t={timestamps[1]}s. "
                "Describe in detail what is visible: any text on screen, diagrams, "
                "objects, people, actions, and any other notable visual elements."
            ),
        },
    ]

    response = img_llm.invoke([HumanMessage(content=content)])
    caption = normalize_message_content(response.content).strip()

    return json.dumps({"timestamp_sec": t, "caption": caption}, indent=2)


TOOLS = [retrieve_audio_transcript, retrieve_image_transcript, caption_frame]
print("Tools:", [t.name for t in TOOLS])

Tools: ['retrieve_audio_transcript', 'retrieve_image_transcript', 'caption_frame']


### 2.4 LLM setup

In [9]:
img_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.2)
orchestrator_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2).bind_tools(TOOLS)
evaluator_llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

### 2.5 Orchestrator system prompt (path-aware)

In [10]:
def make_orchestrator_system(audio_rich: bool) -> str:
    strategy = (
        """AUDIO-RICH DRAFTING STRATEGY:
1. Call `retrieve_audio_transcript` first to gather transcript evidence.
2. If the transcript evidence is incomplete OR the question asks about something
   visual (diagrams, on-screen text, appearance, demonstrations), also call
   `retrieve_image_transcript` for additional caption/OCR context.
3. TIMESTAMP QUERIES — if the question pins a specific time (e.g. 'at 1:56',
   'around the 2-minute mark', 'what is shown here at t=45s') OR asks what the
   professor/speaker is explaining/showing at a moment:
     a. Convert the timestamp to whole seconds (1:56 → 116).
     b. Call `caption_frame(t=<seconds>)` to get a live visual description of
        that moment directly from the video frames.
     c. Combine the frame caption with any transcript/image-index evidence to
        build a complete answer.
4. If after steps 1-3 the answer is still uncertain or shallow, call
   `caption_frame` at the most relevant timestamp found in the evidence.
5. Using everything retrieved, write a DETAILED answer.
6. Output in this EXACT format:

Timestamp: <seconds>
Answer: <your detailed answer>
"""
        if audio_rich else
        """IMAGE-RICH DRAFTING STRATEGY:
1. Call `retrieve_image_transcript` first to gather visual/OCR evidence.
2. If the visual evidence is incomplete or the question asks about speech,
   narration, or concepts explained verbally, also call
   `retrieve_audio_transcript` for transcript context.
3. TIMESTAMP QUERIES — if the question pins a specific time (e.g. 'at 1:56',
   'around the 2-minute mark', 'what is shown here at t=45s') OR asks what the
   professor/speaker is explaining/showing at a moment:
     a. Convert the timestamp to whole seconds (1:56 → 116).
     b. Call `caption_frame(t=<seconds>)` to get a live visual description of
        that moment directly from the video frames.
     c. Combine the frame caption with any image-index/transcript evidence to
        build a complete answer.
4. If after steps 1-3 the answer is still uncertain or shallow, call
   `caption_frame` at the most relevant timestamp found in the evidence.
5. Using everything retrieved, write a DETAILED answer.
6. Output in this EXACT format:

Timestamp: <seconds>
Answer: <your detailed answer>
"""
    )

    return f"""You are VideoRAG, an expert at answering questions about video content.

{strategy}

TOOL REFERENCE:
  retrieve_audio_transcript(query, k)  — semantic search over Whisper transcripts.
  retrieve_image_transcript(query, k)  — semantic search over BLIP captions + OCR.
  caption_frame(t)                     — extract frames at t and t+5 s from the
                                         actual video and generate a fresh visual
                                         description using a multimodal LLM.
                                         Use when the question is timestamp-specific
                                         OR when index tools don't give enough detail.

GLOBAL RULES:
- Always call at least one tool before writing any answer.
- Each tool may be called at most 3 times total across the session.
- Do NOT paste raw JSON or tool outputs into your answer.
- When a timestamp is mentioned in the question, ALWAYS call `caption_frame`
  for that timestamp — do not rely solely on the index tools for such queries.
- If the evidence genuinely contains no relevant information, output:
    Timestamp: N/A
    Answer: Not covered in this video."""





### 2.7 Tool executor

In [13]:
TOOL_MAP = {t.name: t for t in TOOLS}

# Executor limits
MAX_TOOL_CALLS_PER_TOOL = 3   # each tool can be called at most 3 times
MAX_DOCS_PER_CALL       = 10  # k capped at 10

# Observability: list of all tool calls made across the session
retrieved_docs: list[dict] = []


def execute_tool_calls(
    ai_message: AIMessage,
    tool_usage_counter: dict,
) -> tuple[list[ToolMessage], str]:
    """Run every tool call in ai_message; return (ToolMessages, evidence_str).

    Enforces per-tool call limits (MAX_TOOL_CALLS_PER_TOOL).
    Appends every call (successful or skipped) to the global retrieved_docs list
    for observability.
    """
    tool_msgs = []
    evidence = []

    for tc in ai_message.tool_calls:
        name = tc["name"]
        args = tc.get("args", {})

        # Cap k to MAX_DOCS_PER_CALL
        if "k" in args:
            args = dict(args)
            args["k"] = min(int(args["k"]), MAX_DOCS_PER_CALL)

        tool_usage_counter[name] = tool_usage_counter.get(name, 0) + 1

        if tool_usage_counter[name] > MAX_TOOL_CALLS_PER_TOOL:
            warn = (
                f"Tool '{name}' has reached its call limit "
                f"({MAX_TOOL_CALLS_PER_TOOL}). Skipping."
            )
            tool_msgs.append(ToolMessage(content=warn, tool_call_id=tc["id"]))
            evidence.append(f"--- {name}({args}) ---\n{warn}")
            retrieved_docs.append({
                "tool": name, "args": args,
                "status": "skipped (limit reached)", "result": warn,
            })
            continue

        fn = TOOL_MAP.get(name)
        if fn is None:
            warn = f"Tool '{name}' is not available."
            tool_msgs.append(ToolMessage(content=warn, tool_call_id=tc["id"]))
            evidence.append(f"--- {name}({args}) ---\n{warn}")
            retrieved_docs.append({
                "tool": name, "args": args,
                "status": "error (not found)", "result": warn,
            })
            continue

        try:
            result = fn.invoke(args)
            status = "ok"
        except Exception as e:
            result = json.dumps({"error": str(e)})
            status = "error"

        tool_msgs.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
        evidence.append(f"--- {name}({args}) ---\n{result}")
        retrieved_docs.append({
            "tool": name, "args": args,
            "status": status, "result": result,
        })

    return tool_msgs, "\n\n".join(evidence)


# Indexes and routing are built once in the FAISS setup cell above.
# Reuse audio_index, img_index, and AUDIO_RICH here.


### 2.8 Main agent loop

In [14]:
EVALUATOR_SYSTEM = """You are an independent evaluator for a VideoRAG system.
You have no role in generating or retrying answers — you only score.

You receive:
  USER_QUESTION : the original question
  ANSWER        : the answer produced by the system
  EVIDENCE      : the raw retrieved documents used to generate the answer

Score the answer on two axes out of 10:
  Relevance — how directly and completely the answer addresses the question.
  Grounding — how well the answer's core claims are traceable to the evidence.
              Reasonable general-knowledge elaboration is fine; penalise only
              claims that contradict or have no basis in the evidence.

Reply in EXACTLY this format (3 lines, nothing else):
Relevance: <N>/10 (<3-6 word reason>)
Grounding: <N>/10 (<3-6 word reason>)
Quality: <1-2 sentence overall assessment of the answer>"""


def normalize_message_content(content) -> str:
    """Flatten assistant content that may arrive as text, list, or dict."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                if isinstance(item.get("text"), str):
                    parts.append(item["text"])
                elif "content" in item:
                    parts.append(normalize_message_content(item["content"]))
                else:
                    parts.append(json.dumps(item, ensure_ascii=False))
            else:
                parts.append(str(item))
        return "\n".join(p for p in parts if p).strip()
    if isinstance(content, dict):
        if isinstance(content.get("text"), str):
            return content["text"]
        if "content" in content:
            return normalize_message_content(content["content"])
        return json.dumps(content, ensure_ascii=False)
    return str(content)


def evaluate(question: str, answer: str, evidence: str) -> str:
    """Call the evaluator LLM and return its 3-line score block (observability only)."""
    msg = (
        f"USER_QUESTION:\n{question}\n\n"
        f"ANSWER:\n{answer}\n\n"
        f"EVIDENCE:\n{evidence[:6000]}"
    )
    resp = evaluator_llm.invoke([
        SystemMessage(content=EVALUATOR_SYSTEM),
        HumanMessage(content=msg),
    ])
    return normalize_message_content(resp.content).strip()


def run_videorag(question: str, verbose: bool = True) -> str:
    """
    Orchestrator → tool calls → answer → evaluator (observability only).

    - No retries         : evaluator scores but does not influence the answer
    - Max calls per tool : MAX_TOOL_CALLS_PER_TOOL (= 3)
    - Max docs per call  : MAX_DOCS_PER_CALL (= 10)
    - Observability      : retrieved_docs list + evaluator scores printed

    Returns the final formatted answer string.
    """
    retrieved_docs.clear()

    messages: list = [
        SystemMessage(content=make_orchestrator_system(AUDIO_RICH)),
        HumanMessage(content=question),
    ]
    all_evidence = ""
    tool_usage_counter: dict = {}

    # ── Orchestration: tool calls → answer (single pass) ─────────────────────
    while True:
        response = orchestrator_llm.invoke(messages)
        messages.append(response)

        if response.tool_calls:
            tool_msgs, chunk = execute_tool_calls(response, tool_usage_counter)
            messages.extend(tool_msgs)
            all_evidence += chunk + "\n"
            if verbose:
                for tc in response.tool_calls:
                    print(f"  [Tool called] {tc['name']}({tc.get('args', {})})")
        else:
            final_answer = normalize_message_content(response.content).strip()
            break

    if not final_answer:
        final_answer = "Timestamp: N/A\nAnswer: This topic is not covered in this video or could not be found."

    # ── Evaluator: score only, no retry ──────────────────────────────────────
    scores = evaluate(question, final_answer, all_evidence)

    if verbose:
        _print_retrieved_docs()
        print("\n[Evaluator Scores]")
        print(scores)
        print()

    return final_answer


def _print_retrieved_docs():
    """Pretty-print the observability log of all tool calls made."""
    print("\n" + "─" * 50)
    print(f"[Observability] Retrieved docs log ({len(retrieved_docs)} calls):")
    for i, entry in enumerate(retrieved_docs, 1):
        status_icon = "✓" if entry["status"] == "ok" else "✗"
        print(f"  {i}. {status_icon} {entry['tool']}({entry['args']}) → {entry['status']}")
    print("─" * 50)


---
## Part 3 — Run it!

In [16]:
question = "difference between transition metals and actinides?"
answer   = run_videorag(question, verbose=True)

print("\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(answer)

  [Tool called] retrieve_audio_transcript({'k': 3, 'query': 'difference between transition metals and actinides'})

──────────────────────────────────────────────────
[Observability] Retrieved docs log (1 calls):
  1. ✓ retrieve_audio_transcript({'k': 3, 'query': 'difference between transition metals and actinides'}) → ok
──────────────────────────────────────────────────

[Evaluator Scores]
Relevance: 9/10 (direct, comprehensive)  
Grounding: 7/10 (mostly evidence‑based)  
Quality: The answer correctly outlines the main chemical and physical distinctions between transition metals and actinides, drawing on the retrieved material, though a few detailed claims (e.g., oxidation‑state variety, catalytic behavior) are not directly supported by the evidence.


FINAL ANSWER
Timestamp: 30
Answer: Transition metals are the elements in the d‑block of the periodic table (groups 3–12). Their defining feature is a partially filled d‑subshell, which gives them characteristic properties such as high 

In [14]:
question = "What types of immune cells are derived from hematopoietic stem cells?"
answer   = run_videorag(question, verbose=True)

print("\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(answer)

  [Tool called] retrieve_audio_transcript({'k': 5, 'query': 'immune cells derived from hematopoietic stem cells'})
  [Tool called] retrieve_audio_transcript({'k': 10, 'query': 'derived from hematopoietic stem cells'})
  [Tool called] retrieve_image_transcript({'k': 5, 'query': 'hematopoietic stem cells immune cells derived'})

──────────────────────────────────────────────────
[Observability] Retrieved docs log (3 calls):
  1. ✓ retrieve_audio_transcript({'k': 5, 'query': 'immune cells derived from hematopoietic stem cells'}) → ok
  2. ✓ retrieve_audio_transcript({'k': 10, 'query': 'derived from hematopoietic stem cells'}) → ok
  3. ✓ retrieve_image_transcript({'k': 5, 'query': 'hematopoietic stem cells immune cells derived'}) → ok
──────────────────────────────────────────────────

[Evaluator Scores]
Relevance: 9/10 (comprehensive cell list)  
Grounding: 3/10 (evidence lacks lineage details)  
Quality: The answer accurately lists immune cells derived from hematopoietic stem cells, but

In [15]:
question2 = "What are the four key functions of immune cells shown in the video?"
answer2   = run_videorag(question2, verbose=True)
print("\nFINAL ANSWER:\n", answer2)

  [Tool called] retrieve_audio_transcript({'k': 5, 'query': 'four key functions of immune cells'})

──────────────────────────────────────────────────
[Observability] Retrieved docs log (1 calls):
  1. ✓ retrieve_audio_transcript({'k': 5, 'query': 'four key functions of immune cells'}) → ok
──────────────────────────────────────────────────

[Evaluator Scores]
Relevance: 10/10 (direct complete answer)  
Grounding: 9/10 (well‑supported claims)  
Quality: The answer correctly lists the four functions—recognize, react, regulate, remember—and its elaborations are consistent with the transcript evidence.


FINAL ANSWER:
 Timestamp: 270
Answer: The video explains that immune cells carry out four key functions: **recognize**, **react**, **regulate**, and **remember**. Recognizing involves distinguishing self from non‑self (including pathogens, cancerous cells, or damaged tissue). Reacting means deploying appropriate effector actions such as killing pathogens, healing wounds, or clearing debri

In [21]:
question3 = "what is the professors name in the video?"
answer3  = run_videorag(question3, verbose=True)
print("\nFINAL ANSWER:\n", answer3)


  Attempt 1 / 3
  ▶ retrieve_audio_transcript({'k': 5, 'query': 'professor name'})

  ◆ Draft answer:
The professor’s name is **Professor Dave Explains**【t=300s】.

  ✦ Evaluator: PASS

FINAL ANSWER:
 The professor’s name is **Professor Dave Explains**【t=300s】.


In [ ]:
# ── Interactive REPL ──────────────────────────────────────────────────────────
print("VideoRAG — type 'quit' to exit\n")
while True:
    q = input("Question: ").strip()
    if q.lower() in {"quit", "exit", "q"}:
        break
    if not q:
        continue
    print("\nAnswer:", run_videorag(q, verbose=False), "\n")